# 🌟 Twinkle Eval｜數學評測教學

## 專案資訊

| 項目 | 內容 |
|------|------|
| GitHub | https://github.com/ai-twinkle/Eval |
| PyPI | `twinkle-eval` |
| 授權 | MIT |
| Python | ≥ 3.11 |
| Colab 工作目錄 | `/content/Eval/` |

---

## 開場白

數學評測與選擇題評測有根本上的差異。

選擇題的答案是離散的字母（A、B、C、D），只要找到模型輸出的字母就能判斷對錯。但數學題的答案形式千變萬化——`1/2`、`0.5`、`\frac{1}{2}`、`50%` 都可能是同一道題的正確答案，不能用字串直接比對。

Twinkle Eval 的數學評測採用兩層設計：

1. **MathExtractor**：從模型輸出中找出 `\boxed{...}` 內的表達式
2. **MathRulerScorer**：使用 `mathruler` 進行**語意等價**判斷，而非字串完全比對

## 本 Notebook 範圍

| 資料集 | 說明 |
|--------|------|
| GSM8K | 國小到國中程度的文字應用題，答案為整數 |
| AIME 2025 | 美國邀請賽數學競賽題，答案為 0–999 整數 |

> 📓 選擇題評測（Pattern / Box / Logit）請見 **`01_選擇題評測.ipynb`**。

---

# 第一章：環境安裝

數學評測需要額外的語意等價判斷套件。`twinkle-eval[math]` 會一併安裝以下三個工具：

| 套件 | 用途 |
|------|------|
| `mathruler` | 核心語意等價判斷（基於規則的數學比對）|
| `sympy` | 代數符號運算（分數、方程式化簡）|
| `pylatexenc` | LaTeX 表達式解析（將 `\frac{1}{2}` 轉換為可計算形式）|

> ⚠️ 若只安裝 `pip install mathruler`，`sympy` 和 `pylatexenc` **不會被自動安裝**（mathruler 未宣告這些相依性），數學比對在碰到分數或 LaTeX 時會靜默失敗。請務必使用 `twinkle-eval[math]`。

In [ ]:
!pip install -q "twinkle-eval[math]"

In [ ]:
import twinkle_eval
from twinkle_eval.metrics import get_available_methods

print(f"套件版本：{twinkle_eval.__version__}")
print(f"可用評測方法：{get_available_methods()}")

# 確認 math 相依套件都已安裝
for pkg in ["mathruler", "sympy", "pylatexenc"]:
    try:
        __import__(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg} 未安裝，請執行 pip install twinkle-eval[math]")

---

# 第二章：準備資料集

## 數學開放題的欄位格式

數學題不需要選項欄位，只要有題目和答案：

```json
{ "question": "Janet has 3 apples. She gives 1 to Tom. How many does she have?", "answer": "2" }
```

`answer` 欄位填入最終的數字或表達式（不需要加單位）。

## 與選擇題格式的差異

| | 選擇題 | 數學開放題 |
|---|---|---|
| 選項欄位（A/B/C/D）| **必填** | 不需要 |
| answer 格式 | 單一字母（A/B/C/D）| 數字或數學表達式 |
| 評分方式 | 字串完全比對 | 語意等價判斷 |

In [ ]:
import os

if not os.path.exists("/content/Eval"):
    !git clone -q https://github.com/ai-twinkle/Eval.git /content/Eval

os.chdir("/content/Eval")
print(f"工作目錄：{os.getcwd()}")

In [ ]:
import json

math_datasets = {
    "GSM8K（文字應用題）":   "datasets/example/gsm8k/test.jsonl",
    "AIME 2025（競賽數學）": "datasets/example/aime2025/test.jsonl",
}

for name, path in math_datasets.items():
    with open(path) as f:
        lines = f.readlines()
    first = json.loads(lines[0])
    print(f"📂 {name}")
    print(f"   筆數：{len(lines)}　欄位：{list(first.keys())}")
    print(f"   題目：{first['question'][:60]}...")
    print(f"   答案：{first['answer']}")
    print()

---

# 第三章：設定檔——為什麼要關閉 System Prompt？

這是數學評測中最容易踩到的坑。

## 問題根源

Twinkle Eval 預設的 system prompt 是為**選擇題**設計的，內容大意是：「請在最後以 `答案：X` 或 `\boxed{X}` 的格式給出選項字母。」

若對 GSM8K 這種數學文字題也套用這個 system prompt，模型會收到「請給出選項字母」的指令，但題目根本沒有選項字母——模型就會被搞混，可能輸出 `答案：B`（隨機猜一個字母），而不是計算結果。

## 解法：`system_prompt_enabled: false`

在 `dataset_overrides` 中針對數學資料集設定 `system_prompt_enabled: false`，讓模型在沒有選擇題格式指引的情況下自由推理。推理模型在這種情況下，會自然地在推導最後輸出 `\boxed{答案}`，這正是 MathExtractor 要提取的格式。

```yaml
dataset_overrides:
  "datasets/example/gsm8k/":
    evaluation_method: math
    system_prompt_enabled: false   # 關鍵：不套用選擇題 system prompt
```

> ⚠️ **請將下方的 API 資訊替換為你實際的端點與金鑰。**

In [ ]:
API_BASE_URL = "https://your-api-endpoint/v1"
API_KEY      = "sk-your-api-key-here"
MODEL_NAME   = "your-model-name"

In [ ]:
import yaml

math_config = {
    "llm_api": {
        "base_url": API_BASE_URL,
        "api_key": API_KEY,
        "api_rate_limit": -1,
        "max_retries": 3,
        "timeout": 120,   # 數學推理模型回應時間較長，timeout 建議拉高
    },
    "model": {
        "name": MODEL_NAME,
        "temperature": 0.0,
        "max_tokens": 4096,   # 推理模型 chain-of-thought 較長，需要更多 token
    },
    "evaluation": {
        "dataset_paths": [
            "datasets/example/gsm8k/",
            "datasets/example/aime2025/",
        ],
        "evaluation_method": "math",  # 預設使用 math 評測
        "dataset_overrides": {
            "datasets/example/gsm8k/":    {"system_prompt_enabled": False},
            "datasets/example/aime2025/": {"system_prompt_enabled": False},
        },
    },
    "logging": {"level": "INFO"},
}

with open("config_math.yaml", "w", encoding="utf-8") as f:
    yaml.dump(math_config, f, allow_unicode=True, default_flow_style=False)

print("✅ 已建立 config_math.yaml")

---

# 第四章：Math 評測

## 兩層設計

Math 評測由兩個獨立元件組成：

**MathExtractor（提取）**
- 用括號計數法找到輸出中所有的 `\boxed{...}` 和 `\box{...}`
- 若有多個，取**最後一個**（推理模型的最終答案通常在結尾）
- 若完全沒有 `\boxed{}`，取最後一行非空文字作為 fallback

**MathRulerScorer（評分）**
- 第一層：直接呼叫 `mathruler.grade_answer()` 進行語意等價判斷
- 第二層：補強 mathruler 漏掉的大小寫差異（`\FRAC` vs `\frac`）
- 第三層：允許逗號分隔的解集合忽略順序（`1,-2` 等價 `-2,1`）

## 語意等價的重要性

以下都是同一道題「計算 1÷2」的正確答案，字串比對會全部判錯，但語意等價判斷全部通過：

```
"0.5"          ← 小數
"1/2"          ← 分數
"\frac{1}{2}"  ← LaTeX 分數
"50%"          ← 百分比（視 mathruler 版本）
```

In [ ]:
# 先示範 MathRulerScorer 的語意等價判斷
from twinkle_eval.metrics.scorers.math import MathRulerScorer

scorer = MathRulerScorer()

test_cases = [
    ("0.5",            "1/2",           "小數 vs 分數"),
    ("\\frac{1}{2}",   "0.5",           "LaTeX vs 小數"),
    ("42",             "42",            "完全相同"),
    ("42",             "43",            "不相等"),
    ("-2,1",           "1,-2",          "解集合順序不同"),
    (None,             "42",            "預測為 None"),
]

print(f"{'情境':<20} {'預測':<18} {'正解':<10} {'結果'}")
print("-" * 60)
for pred, gold, desc in test_cases:
    result = scorer.score(pred, gold)
    icon = "✅" if result else "❌"
    pred_display = str(pred) if pred is not None else "None"
    print(f"{desc:<20} {pred_display:<18} {gold:<10} {icon}")

In [ ]:
# 執行數學評測
!python -m twinkle_eval.cli --config config_math.yaml

In [ ]:
import glob, json

files = sorted(glob.glob("results/results_*.json"))
if files:
    with open(files[-1]) as f:
        data = json.load(f)

    print(f"{'資料集':<30} {'正確率':>10} {'題數':>8}")
    print("-" * 52)
    for ds, result in data["dataset_results"].items():
        name = ds.rstrip("/").split("/")[-1]
        acc   = result.get("average_accuracy", 0) * 100
        total = result.get("total_count", "-")
        print(f"{name:<30} {acc:>9.2f}% {total:>8}")
    print(f"\n耗時：{data.get('duration_seconds', 0):.1f} 秒")

---

# 第五章：為什麼 Logit 在數學題得 0 分？

這是從 `01_選擇題評測.ipynb` 延伸的重要問題，值得在此完整說明。

## 兩條路徑的根本差異

**Logit 路徑**（選擇題）：
```
對每個選項 X ∈ {A, B, C, ...}：
    呼叫 /v1/completions echo=True → 取 log P(" X" | context)
預測答案 = argmax log-prob
```
這個路徑**從未讓模型生成任何文字**，`llm_output` 永遠是 `null`。

**Math 路徑**（開放題）：
```
呼叫 /v1/chat/completions → 取得模型生成的完整文字
MathExtractor 提取 \boxed{...}
MathRulerScorer 語意等價判斷
```
這個路徑**完全依賴模型生成的文字**。

## 為什麼搭在一起會得 0 分

若在設定檔中讓 Logit 去評測 GSM8K，Evaluator 偵測到 `uses_logprobs=True` 後，會嘗試對題目的「選項」計算 log-prob。但 GSM8K 沒有選項欄位：

1. 沒有選項 → 沒有 continuation 可以打分
2. 所有題目的 `predicted_answer` 都是 `None`
3. `None` 永遠不等於 `"42"` → 全部答錯 → **0 分**

這不是 bug，而是設計邊界：Logit 的前提假設是「存在離散選項」。

## 正確的搭配方式

```
選擇題（TMMLU+、MMLU-Pro）  → pattern / box / logit
數學開放題（GSM8K、AIME）   → math（搭配 system_prompt_enabled: false）
```

同一份設定檔中可以混用，透過 `dataset_overrides` 讓不同資料集走不同路徑——但**不要讓 logit 遇到開放題**。

---

# 第六章：混合評測——選擇題 + 數學同跑

在實際評測場景中，通常會希望一次執行完所有資料集。Twinkle Eval 的 `dataset_overrides` 讓你只需要一份設定檔就能做到：選擇題走 Pattern（或 Box），數學題走 Math，互不干擾。

In [ ]:
import yaml

mixed_config = {
    "llm_api": {
        "base_url": API_BASE_URL,
        "api_key": API_KEY,
        "api_rate_limit": -1,
        "max_retries": 3,
        "timeout": 120,
    },
    "model": {
        "name": MODEL_NAME,
        "temperature": 0.0,
        "max_tokens": 4096,
    },
    "evaluation": {
        "dataset_paths": [
            "datasets/example/tmmluplus/",
            "datasets/example/mmlu_pro/",
            "datasets/example/gsm8k/",
            "datasets/example/aime2025/",
        ],
        "evaluation_method": "pattern",   # 預設：選擇題用 pattern
        "datasets_prompt_map": {
            "datasets/example/mmlu_pro/": "en",
        },
        "dataset_overrides": {
            # 數學資料集：改用 math，關閉選擇題 system prompt
            "datasets/example/gsm8k/":    {"evaluation_method": "math", "system_prompt_enabled": False},
            "datasets/example/aime2025/": {"evaluation_method": "math", "system_prompt_enabled": False},
        },
    },
    "logging": {"level": "INFO"},
}

with open("config_mixed.yaml", "w", encoding="utf-8") as f:
    yaml.dump(mixed_config, f, allow_unicode=True, default_flow_style=False)

print("✅ 已建立 config_mixed.yaml")
print()
print("設定說明：")
print("  tmmluplus  → pattern（繁體中文 system prompt）")
print("  mmlu_pro   → pattern（英文 system prompt）")
print("  gsm8k      → math（無 system prompt）")
print("  aime2025   → math（無 system prompt）")

In [ ]:
# 一次執行所有資料集
!python -m twinkle_eval.cli --config config_mixed.yaml

In [ ]:
import glob, json

files = sorted(glob.glob("results/results_*.json"))
if files:
    with open(files[-1]) as f:
        data = json.load(f)

    print(f"{'資料集':<30} {'評測方法':<12} {'正確率':>10} {'無法解析':>10}")
    print("-" * 66)
    for ds, result in data["dataset_results"].items():
        name   = ds.rstrip("/").split("/")[-1]
        method = result.get("evaluation_method", "-")
        acc    = result.get("average_accuracy", 0) * 100
        up     = result.get("average_unparsed_rate", 0) * 100
        up_str = f"{up:.1f}%" if up > 0 else "-"
        print(f"{name:<30} {method:<12} {acc:>9.2f}% {up_str:>10}")
    print(f"\n耗時：{data.get('duration_seconds', 0):.1f} 秒")

---

## 總結

數學評測的關鍵要點：

1. **安裝**：使用 `twinkle-eval[math]`，不要只裝 `mathruler`
2. **System Prompt**：數學資料集必須設定 `system_prompt_enabled: false`，避免選擇題格式指令干擾推理
3. **評分**：MathRulerScorer 做語意等價判斷，`0.5`、`1/2`、`\frac{1}{2}` 都算對
4. **不要用 Logit**：數學開放題沒有選項，Logit 無法計算，強行搭配會得 0 分
5. **Max Tokens**：推理模型的 chain-of-thought 可能很長，建議設定 `max_tokens: 4096` 以上

---

👈 選擇題評測請見 **`01_選擇題評測.ipynb`**